In [ ]:
#imports
import string # from some string manipulation tasks
import nltk # natural language toolkit
import re # regex
from string import punctuation # solving punctuation problems
from nltk.corpus import stopwords # stop words in sentences
from nltk.stem import WordNetLemmatizer # For stemming the sentence
from nltk.stem import SnowballStemmer # For stemming the sentence
from contractions import contractions_dict # to solve contractions
from autocorrect import Speller #correcting the spellings

In [ ]:
import numpy as np
import pandas as pd
# df_sentiment=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/NYTimes/data/pkl/Sentiment.pkl', compression=None)

In [ ]:
%run "/Users/Farid Srouji/Documents/Research/master/NYTimes/code/Initialization/Initialization.ipynb"

In [ ]:
train_data = df[["VIX Natural Log","Sentiment Natural Log", "Articles"]]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pylab as pl

In [ ]:
df.columns

In [ ]:
train_data['Length'] = train_data['Articles'].apply(len)

In [ ]:
train_data['Length'].plot(kind = 'hist' , bins = 50)

In [ ]:
train_data["Sentiment_ln_return"] = train_data["Sentiment_ln_return"].apply(np.float)#.describe()

In [ ]:
train_data["Sentiment_ln_return"].describe()

In [ ]:
train_data["Sentiment_ln_return"]

In [ ]:
conditions = [
    (train_data["vix_ln_return"] > 0 ),
    (train_data["vix_ln_return"] == 0),
    (train_data["vix_ln_return"] < 0)
    ]
# create a list of the values we want to assign for each condition
values = [-1, 0, 1]

# create a new column and use np.select to assign values to it using our lists as arguments
train_data["Sentiment_ln_return"] = np.select(conditions, values)

In [ ]:
ax = train_data.hist(column = 'Length', by = 'Sentiment_ln_return', bins = 50 , figsize = (10,5));
pl.suptitle('Length via each Sentiment')

In [ ]:
train_data['WordCount'] = train_data['Articles'].apply(len)

# Tokenization

In [ ]:
def sentence_tokenize(text):
    """
    take string input and return a list of sentences.
    use nltk.sent_tokenize() to split the sentences.
    """
    return nltk.sent_tokenize(text)
    
def word_tokenize(text):
    """
    :param text:
    :return: list of words
    """
    return nltk.word_tokenize(text) 
        
def to_lower(text):
    """
    :param text:
    :return:
        Converted text to lower case as in, converting "Hello" to "hello" or "HELLO" to "hello".
    """
    return text.lower() 

# "Garbage in, Garbage Out"

In [ ]:
def autospell(text):
        """
        correct the spelling of the word.
        """
        spells = [spell(w) for w in (nltk.word_tokenize(text))]
        return " ".join(spells) 

def remove_numbers(text):
        """
        take string input and return a clean text without numbers.
        Use regex to discard the numbers.
        """
        output = ''.join(c for c in text if not c.isdigit())
        return output 

def remove_punct(text):
  
        return ''.join(c for c in text if c not in punctuation) 

def remove_stopwords(sentence):
        """
        removes all the stop words like "is,the,a, etc."
        """
        stop_words = stopwords.words('english')
        return ' '.join([w for w in nltk.word_tokenize(sentence) if not w in stop_words]) 

# Normalization

In [ ]:
def lemmatize(text):
    wordnet_lemmatizer = WordNetLemmatizer()

    lemmatized_word = [wordnet_lemmatizer.lemmatize(word)for word in nltk.word_tokenize(text)]
    return " ".join(lemmatized_word) 

In [ ]:
def preprocess(text):
    lower_text = to_lower(text)
    sentence_tokens = sentence_tokenize(lower_text)
    word_list = []
    for each_sent in sentence_tokens:
        lemmatizzed_sent = lemmatize(each_sent)
        clean_text = remove_numbers(lemmatizzed_sent)
        clean_text = remove_punct(clean_text)
        clean_text = remove_stopwords(clean_text)
        word_tokens = word_tokenize(clean_text)
        for i in word_tokens:
            word_list.append(i)
    return word_list


# CountVectorizer

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer 
bow = CountVectorizer(analyzer=preprocess)
from sklearn.feature_extraction.text import TfidfTransformer
tfidf = TfidfTransformer()

# Machine Learning to predict Sentiments



In [ ]:
from sklearn.naive_bayes import MultinomialNB
classifier = MultinomialNB()

In [ ]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ('bow', bow),  # strings to token integer counts
    ('tfidf', tfidf),  # integer counts to weighted TF-IDF scores
    ('classifier', classifier),  # train on TF-IDF vectors w/ Naive Bayes classifier
])

In [ ]:
mean_sen = train_data['Sentiment'].mean()

In [ ]:
# train_data['Sentiment'] = train_data['Sentiment'].apply(lambda x:(mean_sen>x))

In [ ]:
train_data['Sentiment'].mean()

In [ ]:
pipeline.fit(train_data['Articles'], train_data['Sentiment'])
pipeline.score(train_data['Articles'], train_data['Sentiment'])

In [ ]:
all_predictions

In [ ]:
from sklearn.metrics import classification_report

all_predictions = pipeline.predict(train_data['Articles'])

In [ ]:
print(classification_report(train_data['Sentiment'], all_predictions))

In [ ]:
aaa = pd.DataFrame(data=all_predictions)

In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm')

In [ ]:
dummy_text = """Hopefully you are enjoying this tutorial and learning new things"""

my_doc = nlp(dummy_text) 

In [ ]:
print(type(my_doc))

In [ ]:
from spacy.tokenizer import Tokenizer
tokenizer = Tokenizer(nlp.vocab)

In [ ]:
tokens = tokenizer(my_doc.text) 

In [ ]:
list(tokens) 

In [ ]:
from spacy.lang.en.stop_words import STOP_WORDS

print('First twenty stop words: %s' % list(STOP_WORDS)[:20])

In [ ]:
filtered_sent=[]
for word in my_doc:
    if word.is_stop==False:
        filtered_sent.append(word)
print("Filtered Sentence:",filtered_sent) 

In [ ]:
for i in my_doc:
    print("Original Word, ", i, " Lemma form, ", i.lemma_) 

In [ ]:
my_doc = nlp("I am learning well, and progressing good!") 

In [ ]:
for word in my_doc:
    print(word.text,word.pos_) 

In [ ]:
nopunc = []
for word in my_doc:
    if word.pos_ != 'PUNCT':
        nopunc.append(word)
print(nopunc) 

In [ ]:
import string # for punctuations
from spacy.lang.en.stop_words import STOP_WORDS  # removing stopwords 


punctuations = string.punctuation

nlp = spacy.load('en_core_web_sm') #english model for spacy
stop_words = spacy.lang.en.stop_words.STOP_WORDS #stop words 


def preprocess(text):
    #removing the digits
    nonum = ''.join([i for i in text if not i.isdigit()])
    # Creating our token object, which is used to create documents with linguistic annotations.
    tokenized_list = nlp(nonum)

    # Lemmatizing each token and converting each token into lowercase
    tokenized_list = [ word.lemma_.lower().strip() if word.lemma_ != "-PRON-" else word.lower_ for word in tokenized_list ]
    
    # Removing stop words
    tokenized_list = [ word for word in tokenized_list if word not in stop_words and word not in punctuations ]

    # return a preprocessed list of tokens
    return tokenized_list

In [ ]:
preprocess("Hello1 ! I am a good boy") 

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

classifier = MultinomialNB()
tfidf_vector = TfidfVectorizer(tokenizer = preprocess)
# Create pipeline 
pipe = Pipeline([('tfidf',tfidf_vector),
                 ('classifier', classifier)])

# model generation
pipe.fit(train_data['Articles'],train_data['Sentiment'])

In [ ]:
pipe.score(train_data['Articles'],train_data['Sentiment_ln_return'])

In [ ]:
from sklearn.metrics import classification_report
Pred = pipe.predict(train_data['Articles'])
print(classification_report(train_data['Sentiment_ln_return'], Pred)) 

In [ ]:

fig = plt.figure(figsize = (20, 6))
plt.subplot(1, 2, 1)
plt.plot(epoch_tr_acc, label='Train Acc') #training list
plt.plot(epoch_vl_acc, label='Validation Acc') #validation list
plt.title("Accuracy")
plt.legend() 
plt.subplot(1, 2, 2)
plt.plot(epoch_tr_loss, label='Train loss') #training loss
plt.plot(epoch_vl_loss, label='Validation loss') #validation loss
plt.title("Loss")
plt.legend()
plt.grid() #grid 
plt.show() #showing the plots

# NLP Sentiment Analyzer using PyTorch

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import torch #pytorch
import torch.nn as nn #for our model class
import torch.nn.functional as F
from nltk.corpus import stopwords #removing stop words
from collections import Counter #counting the unique numbers
import string 
import re #regex
import seaborn as sns #plotting
import matplotlib.pyplot as plt #plotting
from torch.utils.data import TensorDataset, DataLoader #data prep
from sklearn.model_selection import train_test_split #splitting the dataset for training and testing 

In [ ]:
is_cuda = torch.cuda.is_available()

# If we have a GPU available, we'll set our device to GPU. We'll use this device variable later in our code.
if is_cuda:
    device = torch.device("cuda")
    print("GPU is available")
else:
    device = torch.device("cpu")
    print("GPU not available, CPU used") 

In [ ]:
df_part3['Sentiment_ln_return']

In [ ]:
conditions = [
    (df_part3["vix_ln_return"] > 0 ),
    (df_part3["vix_ln_return"] == 0),
    (df_part3["vix_ln_return"] < 0)
    ]
# create a list of the values we want to assign for each condition
values = ['negative','neutral','positive']

# create a new column and use np.select to assign values to it using our lists as arguments
df_part3["Sentiment"] = np.select(conditions, values)

In [ ]:
df_part3["Sentiment"] 

In [ ]:
X,y = df_part3['Articles'].values,df_part3['Sentiment'].values
x_train,x_test,y_train,y_test = train_test_split(X,y,stratify=y)
print(f'shape of train data is {x_train.shape}')
print(f'shape of test data is {x_test.shape}') 

In [ ]:
dd

In [ ]:
dd = pd.Series(y_train).value_counts()
sns.barplot(x=np.array(['positive','negative','neutral']),y=dd.values)
plt.show()

In [ ]:
def preprocess_string(s):
    # Remove all non-word characters 
    s = re.sub(r"[^\w\s]", '', s)
    # Replace all runs of whitespaces with no space
    s = re.sub(r"\s+", '', s)
    # replace digits with no space
    s = re.sub(r"\d", '', s)

    return s 

In [ ]:
def tockenize(x_train,y_train,x_val,y_val):
    word_list = []

    stop_words = set(stopwords.words('english')) #for removing stop words
    for sent in x_train:
        for word in sent.lower().split(): #lower case sentence splitted in list
            word = preprocess_string(word) #preprocess that word using the above function
            if word not in stop_words and word != '':
                word_list.append(word) #if not stop word then append
  
    corpus = Counter(word_list) #counting the words occurence
    # sorting on the basis of most common words
    corpus_ = sorted(corpus,key=corpus.get,reverse=True)[:1000] 
    # creating a dict
    onehot_dict = {w:i+1 for i,w in enumerate(corpus_)} #this will create a dictionary with their numbers based on most common words
    
    # tockenize
    final_list_train,final_list_test = [],[]
    for sent in x_train:
            final_list_train.append([onehot_dict[preprocess_string(word)] for word in sent.lower().split() if preprocess_string(word) in onehot_dict.keys()]) #picking the number of the word after preprocessing it
    for sent in x_val:
            final_list_test.append([onehot_dict[preprocess_string(word)] for word in sent.lower().split() if preprocess_string(word) in onehot_dict.keys()]) #picking the number of the word after preprocessing it
            
    encoded_train = [1 if label =='positive' else 0 for label in y_train] #encoding the labels
    encoded_test = [1 if label =='positive' else 0 for label in y_val] 
    return np.array(final_list_train), np.array(encoded_train), np.array(final_list_test), np.array(encoded_test),onehot_dict #returning these 4 values
 

In [ ]:
x_train,y_train,x_test,y_test,vocab = tockenize(x_train,y_train,x_test,y_test) 


In [ ]:
print(f'Length of vocabulary is {len(vocab)}') 

In [ ]:
### analyze the length of the sequences so that you can padd them
rev_len = [len(i) for i in x_train]
pd.Series(rev_len).hist()
plt.show()
pd.Series(rev_len).describe() 

In [ ]:
def padding_(sentences, seq_len):
    features = np.zeros((len(sentences), seq_len),dtype=int)
    for ii, review in enumerate(sentences):
        if len(review) != 0:
            features[ii, -len(review):] = np.array(review)[:seq_len]
    return features 

In [ ]:
#we have very less number of reviews with length > 500.
x_train_pad = padding_(x_train,500)
x_test_pad = padding_(x_test,500) 

In [ ]:
# create Tensor datasets
train_data = TensorDataset(torch.from_numpy(x_train_pad), torch.from_numpy(y_train))
valid_data = TensorDataset(torch.from_numpy(x_test_pad), torch.from_numpy(y_test))

# batch size
batch_size = 50

#shuffling
train_loader = DataLoader(train_data, shuffle=True, batch_size=batch_size)
valid_loader = DataLoader(valid_data, shuffle=True, batch_size=batch_size)
 

In [ ]:
class SentimentRNN(nn.Module): #class name
    def __init__(self,no_layers,vocab_size,hidden_dim,embedding_dim,drop_prob=0.5): 
        super(SentimentRNN,self).__init__() #initializing base class
 
        self.output_dim = output_dim #output dimensions
        self.hidden_dim = hidden_dim #hidden dimensions
  
        self.no_layers = no_layers #number of layers
        self.vocab_size = vocab_size #vocabulary size
    
        # embedding 
        self.embedding = nn.Embedding(vocab_size, embedding_dim) #embedding of vocabulary size and embedding dimensions
        
        #lstm
        self.lstm = nn.LSTM(input_size=embedding_dim,hidden_size=self.hidden_dim,
                           num_layers=no_layers, batch_first=True)
 
        
        # dropout layer
        self.dropout = nn.Dropout(0.4) 
    
        # linear and sigmoid layer
        self.fc0 = nn.Linear(self.hidden_dim, 512) #first drop out
        self.fc1 = nn.Linear(512, 256) #1st fc layer
        self.dropout1 = nn.Dropout(0.2) #2nd drop out
        self.fc = nn.Linear(256, output_dim) #2nd fully connected layer
        self.sig = nn.Sigmoid() #for last layer
 

In [ ]:
def forward(self,x,hidden):
        batch_size = x.size(0) #fetching batch size
        # embeddings and lstm_out
        embeds = self.embedding(x)  # shape: Batch x S x Feature   since batch = True
        lstm_out, hidden = self.lstm(embeds, hidden) #passing our embedded layer into lstm
        
        lstm_out = lstm_out.contiguous().view(-1, self.hidden_dim)  #reshaping the lstm output
        
        # dropout and fully connected layer
        out = self.dropout(lstm_out) 
        out = self.fc0(out)
        out = self.dropout1(out)
        out = self.fc1(out)
        out = self.fc(out)
        
        # sigmoid function for binary classification output
        sig_out = self.sig(out) 
        
        # reshape to be batch_size first
        sig_out = sig_out.view(batch_size, -1)

        sig_out = sig_out[:, -1] # get last batch of labels
        
        # return last sigmoid output and hidden state
        return sig_out, hidden
 

In [ ]:
def init_hidden(self, batch_size):
        ''' Initializes hidden state '''
        # Create two new tensors with sizes n_layers x batch_size x hidden_dim,
        # initialized to zero, for hidden state and cell state of LSTM using torch.zeros
        h0 = torch.zeros((self.no_layers,batch_size,self.hidden_dim)).to(device)
        c0 = torch.zeros((self.no_layers,batch_size,self.hidden_dim)).to(device)
        hidden = (h0,c0)
        return hidden 

In [ ]:
class SentimentRNN(nn.Module):
    def __init__(self,no_layers,vocab_size,hidden_dim,embedding_dim,drop_prob=0.5): 
        super(SentimentRNN,self).__init__()
 
        self.output_dim = output_dim #output dimensions
        self.hidden_dim = hidden_dim #hidden dimensions
  
        self.no_layers = no_layers #number of layers
        self.vocab_size = vocab_size #vocabulary size
    
        # embedding 
        self.embedding = nn.Embedding(vocab_size, embedding_dim) #embedding of vocabulary size and embedding dimensions        
        #lstm
        self.lstm = nn.LSTM(input_size=embedding_dim,hidden_size=self.hidden_dim,
                           num_layers=no_layers, batch_first=True)
        
        # dropout layer
        self.dropout = nn.Dropout(0.4)
    
        # linear and sigmoid layer
        self.fc0 = nn.Linear(self.hidden_dim, 512) #first drop out
        self.fc1 = nn.Linear(512, 256) #1st fc layer
        self.dropout1 = nn.Dropout(0.2) #2nd drop out
        self.fc = nn.Linear(256, output_dim) #2nd fully connected layer
        self.sig = nn.Sigmoid() #for last layer
        
    def forward(self,x,hidden):
        batch_size = x.size(0)
        # embeddings and lstm_out
        embeds = self.embedding(x)  # shape: B x S x Feature   since batch = True
        #print(embeds.shape)  #[50, 500, 1000]
        lstm_out, hidden = self.lstm(embeds, hidden)
        
        lstm_out = lstm_out.contiguous().view(-1, self.hidden_dim) 
        
        # dropout and fully connected layer
        out = self.dropout(lstm_out)
        out = self.fc0(out)
        out = self.dropout1(out)
        out = self.fc1(out)
        out = self.fc(out)
        
        # sigmoid function
        sig_out = self.sig(out)
        
        # reshape to be batch_size first
        sig_out = sig_out.view(batch_size, -1)

        sig_out = sig_out[:, -1] # get last batch of labels
        
        # return last sigmoid output and hidden state
        return sig_out, hidden
        
        
        
    def init_hidden(self, batch_size):
        ''' Initializes hidden state '''
        # Create two new tensors with sizes n_layers x batch_size x hidden_dim,
        # initialized to zero, for hidden state and cell state of LSTM
        h0 = torch.zeros((self.no_layers,batch_size,self.hidden_dim)).to(device)
        c0 = torch.zeros((self.no_layers,batch_size,self.hidden_dim)).to(device)
        hidden = (h0,c0)
        return hidden

In [ ]:
no_layers = 4 #4 hidden LSTM stacked layers
vocab_size = len(vocab) + 1 #extra 1 for padding
embedding_dim = 64 # embedding dimensions
output_dim = 1 #single output 1 or 0
hidden_dim = 256 # hidden dimensions
model = SentimentRNN(no_layers,vocab_size,hidden_dim,embedding_dim,drop_prob=0.5)
#moving to gpu if available
model.to(device) 

In [ ]:
print(model) 

In [ ]:
def acc(pred,label):
    pred = torch.round(pred.squeeze()) #remove extra dimensions
    return torch.sum(pred == label.squeeze()).item()  #correct predictions vs label 

In [ ]:
lr=0.001 #learning rate
criterion = nn.BCELoss() #binary cross entropy for binary classification
optimizer = torch.optim.Adam(model.parameters(), lr=lr) #adam optimizer

In [ ]:
clip = 5 #gradient clipping for exploding gradients
epochs = 10 #number of epochs
valid_loss_min = np.Inf #setting loss for minimum i.e infinity epochs
# train for some number of epochs
epoch_tr_loss,epoch_vl_loss = [],[]  #lists for appending per epochs statistics to visualize
epoch_tr_acc,epoch_v_acc = [],[]

In [ ]:
for epoch in range(epochs):
    train_losses = []
    train_acc = 0.0
    model.train()
    # initialize hidden state 
    h = model.init_hidden(batch_size)
    for inputs, labels in train_loader: #training data
        
        inputs, labels = inputs.to(device), labels.to(device)   
        # Creating new variables for the hidden state, otherwise
        # we'd backprop through the entire training history
        h = tuple([each.data for each in h]) 
        
        model.zero_grad() #zero gradients before starting the training
        output,h = model(inputs,h) #forward pass
        
        # calculate the loss and perform backprop
        loss = criterion(output.squeeze(), labels.float()) #criterion loss
        loss.backward() # computes loss
        train_losses.append(loss.item()) #appending the loss
        # calculating accuracy
        accuracy = acc(output,labels) 
        train_acc += accuracy 

        #`clip_grad_norm` helps prevent the exploding gradient problem in RNNs / LSTMs.
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step() #improving the loss via optimizer using back prop (ADAM optimizer)
 
    
        
    val_h = model.init_hidden(batch_size) #initialize the hidden state
    val_losses = []
    val_acc = 0.0
    model.eval()
    for inputs, labels in valid_loader: #validation data
            val_h = tuple([each.data for each in val_h]) #separate variable for hidden state

            inputs, labels = inputs.to(device), labels.to(device) #checking if gpu or not

            output, val_h = model(inputs, val_h) #forward pass
            val_loss = criterion(output.squeeze(), labels.float()) # loss

            val_losses.append(val_loss.item()) #appending loss
            
            accuracy = acc(output,labels)
            val_acc += accuracy
            
    epoch_train_loss = np.mean(train_losses)  #mean of the loss for calculating epoch loss
    epoch_val_loss = np.mean(val_losses) #same for validation loss
    epoch_train_acc = train_acc/len(train_loader.dataset) #acc for epoch
    epoch_val_acc = val_acc/len(valid_loader.dataset) #vali acc for epoch
    epoch_tr_loss.append(epoch_train_loss)
    epoch_vl_loss.append(epoch_val_loss)
    epoch_tr_acc.append(epoch_train_acc)
    epoch_vl_acc.append(epoch_val_acc)
    print(f'Epoch {epoch+1}') 
    print(f'train_loss : {epoch_train_loss} val_loss : {epoch_val_loss}')
    print(f'train_accuracy : {epoch_train_acc*100} val_accuracy : {epoch_val_acc*100}')
    if epoch_val_loss <= valid_loss_min:
        torch.save(model.state_dict(), '../working/state_dict.pt') #saving the model in dictionary in .pt format
        print('Validation loss decreased ({:.6f} --> {:.6f}).  Saving model ...'.format(valid_loss_min,epoch_val_loss))
        valid_loss_min = epoch_val_loss
    print(25*'==') #for epoch ending symbol
 

In [ ]:
fig = plt.figure(figsize = (20, 6))
plt.subplot(1, 2, 1)
plt.plot(epoch_tr_acc, label='Train Acc') #training list
plt.plot(epoch_vl_acc, label='Validation Acc') #validation list
plt.title("Accuracy")
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(epoch_tr_loss, label='Train loss') #training loss
plt.plot(epoch_vl_loss, label='Validation loss') #validation loss
plt.title("Loss")
plt.legend()
plt.grid() #grid 
plt.show() #showing the plots